In [89]:
import pandas as pd
import numpy as np
import sklearn.model_selection as ms
import sklearn.compose as compose
import sklearn.pipeline as pipeline
import sklearn.preprocessing as preprocessing
import sklearn.tree as tree
import sklearn.metrics as metrics


seed = 30

# Load data
df = pd.read_csv("HR_data.csv")

y = df["Frustrated"]
X = df.drop(columns=["Frustrated", "Individual"])
groups = df["Individual"]

# Preprocessing
preprocessor = compose.ColumnTransformer([
    (
        "num",
        preprocessing.StandardScaler(),
        ["HR_Mean", "HR_Median", "HR_std",
         "HR_Min", "HR_Max", "HR_AUC"]
    ),
    (
        "cat",
        preprocessing.OneHotEncoder(handle_unknown="ignore"),
        ["Round", "Phase", "Puzzler", "Cohort"]
    )
])


dt = pipeline.Pipeline([
    ("prep", preprocessor),
    ("model", tree.DecisionTreeClassifier(random_state=seed))
])

# Hyperparameter grid
param_grid = {
    "model__max_depth": [ 3, 5, 7],
    "model__min_samples_split": [5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8, 10]
    
}

# CV SETUP
outer_cv = ms.GroupKFold(n_splits=5)
inner_cv = ms.GroupKFold(n_splits=4)

train_acc_scores = []
test_acc_scores = []

for train_idx, test_idx in outer_cv.split(X, y, groups):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups.iloc[train_idx]

    grid = ms.GridSearchCV(
        estimator=dt,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid.fit(X_train, y_train, groups=groups_train)

    best_model = grid.best_estimator_

   
    train_pred = best_model.predict(X_train)
    test_pred = best_model.predict(X_test)

    train_acc_scores.append(
        metrics.accuracy_score(y_train, train_pred)
    )

    test_acc_scores.append(
        metrics.accuracy_score(y_test, test_pred)
    )

print("Mean Training Accuracy:", round(np.mean(train_acc_scores), 4))
print("Mean Test Accuracy:", round(np.mean(test_acc_scores), 4))


Mean Training Accuracy: 0.6265
Mean Test Accuracy: 0.2111


In [90]:
X_encoded = pd.get_dummies(
    X,
    columns=['Round', 'Phase', 'Puzzler', 'Cohort']
)

print("Number of features:")
print(len(X_encoded.columns)-1)
print(X_encoded.columns.tolist())

Number of features:
17
['Unnamed: 0', 'HR_Mean', 'HR_Median', 'HR_std', 'HR_Min', 'HR_Max', 'HR_AUC', 'Round_round_1', 'Round_round_2', 'Round_round_3', 'Round_round_4', 'Phase_phase1', 'Phase_phase2', 'Phase_phase3', 'Puzzler_0', 'Puzzler_1', 'Cohort_D1_1', 'Cohort_D1_2']
